In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
df2=pd.read_excel("/content/drive/MyDrive/IBPS/final_date_edited.xlsx")

In [46]:
df=df2[:15]

In [50]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150430 entries, 0 to 150429
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   CNR                   150430 non-null  object
 1   bail_type             150430 non-null  object
 2   is_withdrawal         150430 non-null  object
 3   age                   150430 non-null  object
 4   health_issues         3686 non-null    object
 5   past_criminal_record  150430 non-null  object
 6   statutes              150430 non-null  object
 7   outcome               150430 non-null  object
 8   date_of_arrest        150311 non-null  object
 9   date_of_judgement     150286 non-null  object
 10  case_details_length   150430 non-null  int64 
 11  case_detals           150430 non-null  object
dtypes: int64(1), object(11)
memory usage: 13.8+ MB


In [6]:
df.head()

,CNR,bail_type,is_withdrawal,age,health_issues,past_criminal_record,statutes,outcome,date_of_arrest,date_of_judgement,case_details_length,case_detals
0,KLHC010003852010,Anticipatory-Bail,No,40,NaN,No,"['Section 341 IPC', 'Section 376 IPC', 'Sectio...",Not Granted,Unknown,26-07-2010,810,The accused are charged with offences related ...
1,KLHC010008822010,Regular-Bail,No,30,NaN,No,"['Section 143 IPC', 'Section 147 IPC', 'Sectio...",Not Granted,05-01-2010,08-03-2010,730,The accused are charged with offences related ...
2,HCBM010413142013,Anticipatory-Bail,No,Unknown,NaN,No,"['Section 465 IPC', 'Section 467 IPC', 'Sectio...",Not Granted,Unknown,23-10-2013,579,The applicant apprehends arrest in furtherance...
3,HCBM010130592019,Anticipatory-Bail,No,Unknown,NaN,Yes,"['Section 415 IPC', 'Section 420 IPC', 'Sectio...",Granted,Unknown,Unknown,517,The applicants are apprehending arrest in a ca...
4,HCBM010049052020,Anticipatory-Bail,No,Unknown,NaN,No,"['Section 467 IPC', 'Section 468 IPC', 'Sectio...",Not Granted,Unknown,26-02-2020,416,"The applicant, Eknath Kondiba Bhujbal, is impl..."


In [ ]:
# ✅ 1. Install required libraries
!pip install transformers accelerate --quiet

In [20]:
# ✅ 2. Import dependencies
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import pandas as pd
from tqdm import tqdm

In [35]:
# Load the larger FLAN-T5 model
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
from transformers import PreTrainedTokenizerFast

In [45]:
# Your refined categories
crime_categories = [
    "Murder",
    "Attempt to murder",
    "Rape",
    "Sexual assault",
    "Domestic violence",
    "Kidnapping and abduction",
    "Assault and battery",
    "Theft",
    "Robbery",
    "Burglary",
    "Arson",
    "Fraud and forgery",
    "Cybercrime",
    "Drug offences",
    "Crimes against property",
    "Crimes against children",
    "Statutory offences",
    "Public order offences",
    "White-collar crimes"
]

# Setup the text2text-generation pipeline with the loaded model
device = 0 if model.device.type == "cuda" else -1
clf = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=0)

# Few-shot examples (OPTIONAL for better accuracy)
few_shot_examples = """
Case: The accused forged documents to claim property under false identity.
Crime categories: Fraud and forgery, Crimes against property

Case: A man was caught carrying narcotic substances without a license.
Crime categories: Drug offences

Case: The accused broke into a house and stole cash and jewelry.
Crime categories: Burglary, Theft
"""

def classify_case(text, max_new_tokens=64,max_tokens=512):
    prompt = f"""You are a legal assistant. Classify the following bail case into one or more of these crime categories (comma-separated):
{", ".join(crime_categories)}.

{few_shot_examples}

Case: {text}
Crime categories:"""

    # Use max_new_tokens to control output length; you can adjust or add truncation on input if needed
    result = clf(prompt, max_new_tokens=60, do_sample=False)[0]['generated_text']

    # Clean up result: split by comma and strip whitespace
    categories = [cat.strip() for cat in result.strip().split(",") if cat.strip()]
    return categories

Device set to use cuda:0


In [48]:
tqdm.pandas()
# Run classification on each row
df['Crime'] = df['case_detals'].progress_apply(classify_case)

df.to_csv("bail_with_crime_classification.csv", index=False)


  0%|          | 0/15 [00:00<?, ?it/s]

/tmp/ipython-input-48-914427222.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Crime'] = df['case_detals'].progress_apply(classify_case)


In [49]:
df[['case_detals', 'Crime']]

,case_detals,Crime
0,The accused are charged with offences related ...,[Sexual assault]
1,The accused are charged with offences related ...,[Kidnapping and abduction]
2,The applicant apprehends arrest in furtherance...,[Fraud and forgery]
3,The applicants are apprehending arrest in a ca...,[White-collar crimes]
4,"The applicant, Eknath Kondiba Bhujbal, is impl...",[Fraud and forgery]
5,"The incident occurred on 15th July 2017, invol...",[Assault and battery]
6,"The applicant's husband, Alok Agrawal, is accu...",[White-collar crimes]
7,"The accused, in furtherance of common intentio...",[Fraud and forgery]
8,The applicants are accused in connection with ...,[White-collar crimes]
9,The de facto complainant was in love with one ...,[Assault and battery]


In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.30.2 accelerate --quiet
!pip uninstall torch torchvision -y
!pip install torch==2.0.1 torchvision==0.15.2 --quiet

Found existing installation: transformers 4.53.2
Uninstalling transformers-4.53.2:
  Successfully uninstalled transformers-4.53.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 105.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.30.2 which is incompatible.
Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 917.3 kB/s

In [28]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm.auto import tqdm
import torch

# Define the model and tokenizer (multi-label capable DistilBERT variant example)
model_name = "joeddav/distilbert-base-uncased-go-emotions-student"  # Multi-label example; replace with your best fit

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Setup classification pipeline with batching and GPU if available
device = 0 if torch.cuda.is_available() else -1
clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    return_all_scores=True,
)


tokenizer_config.json:   0%|          | 0.00/421 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (953 > 512). Running this sequence through the model will result in indexing errors


RuntimeError: The size of tensor a (953) must match the size of tensor b (512) at non-singleton dimension 1

In [33]:
# Define your crime categories exactly as labels expected by the model or map model labels to your categories
crime_categories = [
    "Murder",
    "Attempt to murder",
    "Rape",
    "Sexual assault",
    "Domestic violence",
    "Kidnapping and abduction",
    "Assault and battery",
    "Theft",
    "Robbery",
    "Burglary",
    "Arson",
    "Fraud and forgery",
    "Cybercrime",
    "Drug offences",
    "Crimes against property",
    "Crimes against children",
    "Statutory offences",
    "Public order offences",
    "White-collar crimes",
    "Misdemeanor/Minor offences"
]

# Print raw model outputs to inspect labels and scores
def debug_classify_case(text):
    outputs = clf(text, truncation=True, max_length=512, return_all_scores=True)
    print(outputs)  # Inspect this output for labels and scores
    threshold = 0.1
    labels_over_threshold = [
        out['label'] for out in outputs[0] if out['score'] > threshold
    ]
    return labels_over_threshold

# Customize the multi-label classifier to your crime categories and threshold
def classify_case(text, threshold=0.3):
    outputs = clf(text, truncation=True, max_length=512)
    # outputs is a list of dicts: [{'label': 'Label1', 'score': 0.95}, {...}]
    labels_over_threshold = [
        out['label'] for out in outputs[0] if out['score'] > threshold
    ]
    return labels_over_threshold

# Apply classification on dataset with progress bar
tqdm.pandas()
df['Crime'] = df['case_detals'].progress_apply(debug_classify_case)

# Save output
df.to_csv("bail_with_multicrimes.csv", index=False)

  0%|          | 0/15 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


[[{'label': 'admiration', 'score': 0.01051140483468771}, {'label': 'amusement', 'score': 0.005396380554884672}, {'label': 'anger', 'score': 0.043887924402952194}, {'label': 'annoyance', 'score': 0.03182318061590195}, {'label': 'approval', 'score': 0.07684440910816193}, {'label': 'caring', 'score': 0.05241028591990471}, {'label': 'confusion', 'score': 0.05907505005598068}, {'label': 'curiosity', 'score': 0.05330708250403404}, {'label': 'desire', 'score': 0.08971182256937027}, {'label': 'disappointment', 'score': 0.09082134813070297}, {'label': 'disapproval', 'score': 0.07483518868684769}, {'label': 'disgust', 'score': 0.036479830741882324}, {'label': 'embarrassment', 'score': 0.047636017203330994}, {'label': 'excitement', 'score': 0.04575938731431961}, {'label': 'fear', 'score': 0.016326522454619408}, {'label': 'gratitude', 'score': 0.006024440750479698}, {'label': 'grief', 'score': 0.03756396472454071}, {'label': 'joy', 'score': 0.005802292376756668}, {'label': 'love', 'score': 0.00497

/tmp/ipython-input-33-1046369532.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Crime'] = df['case_detals'].progress_apply(debug_classify_case)


In [34]:
df[['case_detals', 'Crime']]

,case_detals,Crime
0,The accused are charged with offences related ...,[]
1,The accused are charged with offences related ...,[]
2,The applicant apprehends arrest in furtherance...,[caring]
3,The applicants are apprehending arrest in a ca...,[]
4,"The applicant, Eknath Kondiba Bhujbal, is impl...",[]
5,"The incident occurred on 15th July 2017, invol...",[approval]
6,"The applicant's husband, Alok Agrawal, is accu...",[desire]
7,"The accused, in furtherance of common intentio...",[confusion]
8,The applicants are accused in connection with ...,[caring]
9,The de facto complainant was in love with one ...,[desire]
